In [ ]:
import pandas as pd

chunks = pd.read_csv("flights.csv", chunksize=500_000, dtype=str, low_memory=False)
sample = pd.concat(c.sample(frac=0.0086, random_state=42) for c in chunks)
print("Avant nettoyage :", len(sample))

cles = ["YEAR","MONTH","DAY","DAY_OF_WEEK","AIRLINE","ORIGIN_AIRPORT",
        "DESTINATION_AIRPORT","SCHEDULED_DEPARTURE","SCHEDULED_TIME",
        "DISTANCE","CANCELLED","DIVERTED"]

sample = sample.dropna(subset=cles)

# Vol ni annulé ni dérouté mais sans retard à l'arrivée : anormal
anormal = (sample.CANCELLED == "0") & (sample.DIVERTED == "0") & sample.ARRIVAL_DELAY.isna()
sample = sample[~anormal]

# 2. Remplir les vides "normaux" avec une valeur explicite
sample["CANCELLATION_REASON"] = sample["CANCELLATION_REASON"].fillna("N")  # N = non annulé
sample["TAIL_NUMBER"] = sample["TAIL_NUMBER"].fillna("UNKNOWN")
sample = sample.fillna("0")  # causes de retard, horaires et durées restants

# 3. Vérification
print("Après nettoyage :", len(sample))
print("Valeurs manquantes restantes :", sample.isna().sum().sum())  # doit afficher 0

sample.to_csv("flights_sample.csv", index=False)

In [ ]:
flight=pd.read_csv("flights_sample.csv")
flight.shape